# Directly-follows graph: demo on the running example

Page: `docs/pages/directly-follows-graph.md` (status: suggestion, 2026-11-14); user-facing description: module note `overview/module/Directly-Follows Graph Page.md` (outside this repo).
Code: package `ocelescope_module_location_based_ocpm.dfg`; this notebook only imports and calls its functions, so a change in the package shows up here on the next run.
Further detail: the docstrings (e.g. `help(variant_filter)`).

What the notebook does, one step per section (the order of the page, Nina 2026-11-15):
1. loads the running example and derives the locations (for $HU$, $M$, $P$),
2. builds the object traces of the selected object types (`object_traces`),
3. projects the traces on the selected activities (`type_activity_counts`, `default_activities`, `project_traces`),
4. applies the variant filter (`DEFAULT_VARIANT_THRESHOLD` = 80 %, `variant_filter`; deselected activities: `added_activity_thresholds`),
5. counts the directly-follows arcs and applies the arc threshold, the arc count and the arcs removed by hand (`directly_follows_arcs`, `arc_share_filter`, `arc_count_filter`, `remove_arcs`),
6. keeps every arc on a start-to-end path ("Stay connected", `stay_connected`),
7. computes the activity nodes (`shown_activities`, `activity_frequencies`, `involved_object_types`),
8. runs everything in one call, as the page does (`resolve_selection`, `directly_follows_graph`).

In [ ]:
import json
from pathlib import Path

import polars as pl
from ocelescope import OCEL

from ocelescope_module_location_based_ocpm.derivation.config import DerivationConfig
from ocelescope_module_location_based_ocpm.derivation.pipeline import derive
from ocelescope_module_location_based_ocpm.dfg.arcs import arc_count_filter, arc_share_filter, directly_follows_arcs, remove_arcs
from ocelescope_module_location_based_ocpm.dfg.connectivity import stay_connected
from ocelescope_module_location_based_ocpm.dfg.graph import (
    DEFAULT_VARIANT_THRESHOLD,
    DfgRequest,
    TypeThresholds,
    default_selection,
    directly_follows_graph,
    log_frames,
    resolve_selection,
)
from ocelescope_module_location_based_ocpm.dfg.nodes import activity_frequencies, involved_object_types, shown_activities
from ocelescope_module_location_based_ocpm.dfg.projection import default_activities, project_traces, type_activity_counts
from ocelescope_module_location_based_ocpm.dfg.traces import considered_objects, object_traces
from ocelescope_module_location_based_ocpm.dfg.variants import added_activity_thresholds, object_variants, variant_filter, variant_ranking

EXAMPLE = Path.cwd().parent / "data" / "running_example"
pl.Config.set_tbl_rows(40)

## 1. Log, configuration and frames: `log_frames(ocel, derivation)`

The graph needs the e2o relations with object types, the events with the flags $M$, $T$, $P$ (for the AS4 tie order), all objects, the handling units $HU$ and the processing operations $P$. `log_frames` reads them from the log and its derivation into a `LogFrames` object.

In [ ]:
ocel = OCEL.read(EXAMPLE / "running_example.sqlite")
config = DerivationConfig.model_validate(json.loads((EXAMPLE / "configuration.json").read_text()))
tables = {"locations.csv": pl.read_csv(EXAMPLE / "inputs" / "locations.csv", infer_schema=False)}
derivation = derive(ocel, config, tables)
frames = log_frames(ocel, derivation)
frames.e2o

## 2. Traces: `considered_objects(objects, object_types, handling_units)`, `object_traces(e2o, events, objects, processing)`

| parameter | meaning |
|---|---|
| `object_types` | the selected object types (default on the page: `default_selection`, the types with handling units) |
| `handling_units` | $HU$; a type with handling units uses only its handling units, other types all their objects |
| `processing` | $P$ as event ids, for the AS4 tie rank (None = $T$) |

How it works: joins e2o (an object linked twice to an event counts once) with the objects and events and sorts by object, time, DFG boundary rank (a transformation event first at the object's first timestamp, last at its last timestamp; `boundary_rank`), AS4 tie rank, event id; `position` is the 1-based position in the object's trace.

In [ ]:
selection = default_selection(frames.objects, frames.handling_units.to_list())
types = [s.object_type for s in selection]
objects = considered_objects(frames.objects, types, frames.handling_units.to_list())
traces = object_traces(frames.e2o, frames.events, objects, processing=frames.processing.to_list())
traces

## 3. Projection on the selected activities: `type_activity_counts`, `default_activities(counts, object_types)`, `project_traces(traces, included)`

As in Ocelescope's DFG discovery, the events of deselected activities are removed from the traces before anything else: $\langle a, b, c \rangle$ without $b$ becomes $\langle a, c \rangle$. The selection is per object type (`included`: type → activities to keep; a missing type keeps all).
`type_activity_counts` counts the events of every activity per type (only the type's considered objects), most first, ties by first occurrence. The default of a type (`default_activities`) is its most frequent 40 % of activities, at least 3 and at most 10 (coil has 7 activities → 3), plus every activity preselected for one of the selected handling-unit types (`shared_from`) that the type has events of: coil also gets roll from slab.

In [ ]:
counts = type_activity_counts(frames.e2o, frames.events, frames.objects, frames.handling_units.to_list())
included = default_activities(counts, types, shared_from=types)  # the default types are the handling-unit types
counts, included

In [ ]:
projected = project_traces(traces, included)
projected.filter(pl.col("object") == "c1")

### 3a. Event categories: `event_categories(events, vehicles, storage_units)`, `remove_categories(traces, categories, excluded)`

The toolbar's chips Movements / Move to vehicle / Move to storage / Processing / Transforming (all but Move to vehicle and Move to storage selected by default, Nina 2026-11-16). An event is removed from the traces (before the activity projection) only when all categories it belongs to are deselected. `log_frames` computes `frames.categories`; here Movements is deselected, so the crane moves to A2 go, while "load truck" (also Move to vehicle, $uloc(e) \in V$) stays.

In [ ]:
from ocelescope_module_location_based_ocpm.dfg.categories import category_counts, remove_categories

category_counts(frames.categories), project_traces(remove_categories(traces, frames.categories, ["movement"]), included).filter(pl.col("object") == "c1")

## 4. Variant filter: `variant_filter(traces, thresholds)`

The variant of an object is the sequence of activities of its projected trace (`object_variants`). `variant_ranking` counts the objects per variant and object type, most frequent first (ties by first occurrence in the log, then alphabetically), with the objects covered by the variants ranked before (`covered_before`).
A variant of type $ot$ is kept if `covered_before` $< \pi_{ot}/100 \cdot |O_{ot}|$; `thresholds` maps object type → $\pi_{ot}$ in percent (missing = 100, all variants).
The page's default is 80 % for every type (`DEFAULT_VARIANT_THRESHOLD`, Nina 2026-11-16). Here every coil has its own variant: the first three cover 60 %, so 80 % keeps four (4 coils).

In [ ]:
variants = object_variants(projected)
ranking = variant_ranking(variants)
default = DEFAULT_VARIANT_THRESHOLD
default, ranking

In [ ]:
kept_traces = variant_filter(projected, {t: default for t in types})
kept_traces["object"].unique().sort().to_list()

Activities on no included trace: `activity_variant_thresholds(ranking)` gives per type the smallest whole variant threshold that puts an activity on an included trace (the lower part of the page's activity list). For a deselected activity, `added_activity_thresholds(traces, variants, included)` gives the threshold it would need if it were selected as well (the traces projected on the selected activities plus this one; Nina 2026-11-16). Column `needed_exact`: the same with up to 3 decimals, as the page shows it in the unfold (e.g. 2 of 3 objects before: 66.667 % instead of 67 %; Nina 2026-11-18).

In [ ]:
from ocelescope_module_location_based_ocpm.dfg.variants import activity_variant_thresholds

activity_variant_thresholds(ranking), added_activity_thresholds(traces, variants, included)

## 5. Arcs: `directly_follows_arcs(traces)`, `arc_share_filter(arcs, thresholds)`, `arc_count_filter(arcs, counts)`, `remove_arcs(arcs, removed)`

Arcs per object type with `count` (traversals), `objects` (distinct objects) and `first_time` (first traversal); a null `source` is the start node $\triangleright_{ot}$, a null `target` the end node $\square_{ot}$.
`arc_share_filter` (the arc threshold, as Ocelescope's DFG filter) ranks all arcs of a type, start and end arcs included, by `objects`, then `count`, and keeps the first $\lceil \alpha_{ot}/100 \cdot n \rceil$ plus all arcs tied with the last kept one (default on the page 50 %).
`arc_count_filter` (like the slider of Ocelescope's DFG viewer) keeps the first $N$ arcs per type in the order objects, count, first occurrence, source, target, exactly $N$; with `total`, the types without an own count share one limit (the page's default "General" mode, Nina 2026-11-16).
`remove_arcs` drops the arcs removed by hand (right click on the page). This demo uses all activities of coil, so that there are more arcs.

In [ ]:
all_activities = frames.events["activity"].unique().to_list()
coil_traces = project_traces(traces, {"coil": all_activities, "slab": all_activities})
arcs = directly_follows_arcs(coil_traces)
arcs

In [ ]:
arc_share_filter(arcs, {"coil": 20})

In [ ]:
kept_arcs = remove_arcs(arc_count_filter(arcs, {"coil": 2, "slab": 1}), [("coil", "quality check", None)])
kept_arcs

## 6. Stay connected: `stay_connected(base, kept)`

Optional on the page (default on): every shown arc lies on a path from the start to the end node of its type. Arcs that the arc threshold or the arc count removed are added back where needed, least frequent ones first taken out (`connecting` = true, dashed on the page); `base` is all arcs of the included traces without the arcs removed by hand, so these never come back.

In [ ]:
stay_connected(remove_arcs(arcs, [("coil", "quality check", None)]), kept_arcs)

## 7. Activity nodes: `shown_activities(arcs)`, `activity_frequencies(traces, activities)`, `involved_object_types(traces, activities, typed_e2o)`

Shown: the activities touched by a shown arc. Frequency: the events of the activity in the included (projected, variant-filtered) traces. Involved object types: per activity and object type (selected or not) the events and objects; the page draws the unselected ones as circles. The running example has only the handling-unit types coil and slab, so no circles appear here (`tests/test_dfg.py` has a log with a truck).

In [ ]:
activities = shown_activities(arcs)
activity_frequencies(coil_traces, activities)

In [ ]:
involved_object_types(coil_traces, activities, frames.e2o)

## 8. Everything at once: `resolve_selection(request, objects, handling_units)`, `directly_follows_graph(frames, selection, traces, activities, stay, removed, excluded_categories, arc_total)`

What the page's route does: `resolve_selection` turns the page's request (`DfgRequest`: selected types or None for `default_selection`, general `variant_threshold` (None = default), `arc_threshold` and `arc_count`, own values per type in `type_thresholds`) into one `TypeSetting` per selected type. `directly_follows_graph` also takes the activities per type (`activities`; missing types get the default), "Stay connected" (`stay`), the arcs removed by hand (`removed`), the deselected event categories (`excluded_categories`) and the general Arcs (display) limit (`arc_total`), and returns all object types with what is kept of them, the activity nodes, the shown arcs, the activity list (`activity_options`) and the default variant threshold. `traces` can be passed from a cache.

In [ ]:
request = DfgRequest(
    type_thresholds=[TypeThresholds(object_type="coil", variant_threshold=100, arc_count=3)],
    activities={"coil": all_activities},
)
selection = resolve_selection(request, frames.objects, frames.handling_units.to_list())
graph = directly_follows_graph(frames, selection, activities=request.activities, stay=request.stay_connected)
graph.variant_default, pl.DataFrame([t.model_dump() for t in graph.object_types])

In [ ]:
pl.DataFrame([a.model_dump() for a in graph.arcs])